In [0]:
%pip install great-expectations==0.18.21

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## Bronze Great Expectations
Keeps Bronze focused on ingestion health only: lineage, batch viability, malformed payload rescue, and raw schema presence needed for Silver transforms.

In [0]:
from datetime import datetime, timezone
import os

import great_expectations as gx
from great_expectations.checkpoint import Checkpoint
from pyspark.sql import functions as F
from pyspark.sql import types as T

In [0]:
# CSV_BRONZE_PATH = dbutils.jobs.taskValues.get(
#     taskKey="bronze_ingest",
#     key="csv_bronze_path",
#     debugValue=""
# )
# JSON_BRONZE_PATH = dbutils.jobs.taskValues.get(
#     taskKey="bronze_ingest",
#     key="json_bronze_path",
#     debugValue=""
# )

STORAGE_ACCOUNT = "hantstorageaccount"
LAKEHOUSE_CONTAINER = "lakehouse"

GX_ROOT_DIR = "/local_disk0/tmp/ge/bronze"
LOCAL_DATA_DOCS_DIR = "/local_disk0/tmp/ge/bronze_data_docs"
LOCAL_DATA_DOCS_URI = f"file:{LOCAL_DATA_DOCS_DIR}"
DATA_DOCS_SITE_NAME = "bronze_local_site"
PIPELINE_LAYER = "bronze"

CATALOG_NAME = "hant-catalog"
SCHEMA_NAME = "invoice"

FILESTORE_DATA_DOCS_DBFS_PATH = "dbfs:/FileStore/great_expectations/bronze/"
FILESTORE_DATA_DOCS_BROWSER_PATH = "/files/great_expectations/bronze/index.html"

STORAGE_DATA_DOCS_LATEST_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/monitoring/data_docs/bronze/latest/"
STORAGE_DATA_DOCS_RUNS_ROOT = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/monitoring/data_docs/bronze/runs/"

CSV_BRONZE_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/bronze/csv_invoice_raw/"
JSON_BRONZE_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/bronze/json_invoice_raw/"

VALID_CSV_BRONZE_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/bronze/validated/csv_raw/"
VALID_JSON_BRONZE_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/bronze/validated/json_raw/"
QUARANTINE_CSV_BRONZE_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/bronze/ge/quarantine/csv_raw/"
QUARANTINE_JSON_BRONZE_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/bronze/ge/quarantine/json_raw/"
WARNING_CSV_BRONZE_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/bronze/ge/warning/csv_raw/"
WARNING_JSON_BRONZE_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/bronze/ge/warning/json_raw/"
BRONZE_ISSUE_LOG_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/bronze/ge/issue_log/"

GE_RUNTIME_METRICS_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/monitoring/ge_runtime_metrics/"
GE_RUNTIME_METRICS_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.batch_ge_runtime_metrics"

REQUIRED_CSV_COLUMNS = [
    "_source_file", "_ingest_ts", "InvoiceId", "OrderDate", "CustomerName",
    "ShipPostalCode", "ShipCity", "ShipState", "ShipCountry", "ShipMode",
    "BalanceDue", "SubTotal", "DiscountPercent", "DiscountAmount",
    "ShippingAmount", "InvoiceTotal", "OrderId", "ProductName",
    "SubCategory", "Category", "ProductId", "Quantity", "UnitPrice",
    "ItemSubTotal",
]
REQUIRED_JSON_COLUMNS = ["_source_file", "_ingest_ts", "analyzeResult"]

ISSUE_SCHEMA = T.StructType(
    [
        T.StructField("layer", T.StringType(), False),
        T.StructField("dataset", T.StringType(), False),
        T.StructField("rule_id", T.StringType(), False),
        T.StructField("severity", T.StringType(), False),
        T.StructField("_source_file", T.StringType(), True),
        T.StructField("InvoiceId", T.StringType(), True),
        T.StructField("LineNumber", T.StringType(), True),
        T.StructField("dq_reason", T.StringType(), False),
        T.StructField("issue_ts", T.TimestampType(), False),
    ]
)

GE_RUNTIME_SCHEMA = T.StructType([
    T.StructField("layer", T.StringType(), False),
    T.StructField("suite_name", T.StringType(), False),
    T.StructField("run_id", T.StringType(), True),
    T.StructField("started_at", T.TimestampType(), False),
    T.StructField("ended_at", T.TimestampType(), False),
    T.StructField("runtime_seconds", T.DoubleType(), False),
    T.StructField("rows_evaluated", T.LongType(), True),
    T.StructField("rows_per_second", T.DoubleType(), True),
    T.StructField("evaluated_expectations", T.LongType(), True),
    T.StructField("successful_expectations", T.LongType(), True),
    T.StructField("failed_expectations", T.LongType(), True),
    T.StructField("expectation_success_percent", T.DoubleType(), True),
    T.StructField("validation_success", T.BooleanType(), True),
    T.StructField("recorded_at", T.TimestampType(), False),
])

In [0]:
def create_gx_context():
    os.makedirs(GX_ROOT_DIR, exist_ok=True)
    return gx.get_context(context_root_dir=GX_ROOT_DIR)


def add_or_update_spark_datasource(context, datasource_name):
    try:
        return context.sources.add_or_update_spark(name=datasource_name)
    except AttributeError:
        context.add_datasource(
            datasource_name,
            class_name="Datasource",
            execution_engine={"class_name": "SparkDFExecutionEngine"},
            data_connectors={
                "runtime_data_connector": {
                    "class_name": "RuntimeDataConnector",
                    "batch_identifiers": ["default_identifier_name"],
                }
            },
        )
        return context.get_datasource(datasource_name)


def add_or_update_dataframe_asset(datasource, asset_name, dataframe):
    try:
        asset = datasource.get_asset(asset_name)
    except Exception:
        asset = datasource.add_dataframe_asset(name=asset_name)
    return asset.build_batch_request(dataframe=dataframe)


def ensure_data_docs_site(context):
    try:
        context.delete_data_docs_site(DATA_DOCS_SITE_NAME)
    except Exception:
        pass

    context.add_data_docs_site(
        site_name=DATA_DOCS_SITE_NAME,
        site_config={
            "class_name": "SiteBuilder",
            "store_backend": {
                "class_name": "TupleFilesystemStoreBackend",
                "base_directory": LOCAL_DATA_DOCS_DIR,
            },
            "site_index_builder": {"class_name": "DefaultSiteIndexBuilder"},
        },
    )


def build_dataframe_checkpoint(context, batch_request, suite_name, checkpoint_name):
    checkpoint = Checkpoint(
        name=checkpoint_name,
        data_context=context,
        validations=[
            {
                "batch_request": batch_request,
                "expectation_suite_name": suite_name,
            }
        ],
        action_list=[
            {
                "name": "store_validation_result",
                "action": {"class_name": "StoreValidationResultAction"},
            },
            {
                "name": "update_data_docs",
                "action": {"class_name": "UpdateDataDocsAction"},
            },
        ],
        run_name_template="%Y%m%dT%H%M%S_bronze_ge",
    )
    context.add_or_update_checkpoint(checkpoint=checkpoint)
    return checkpoint


def _safe_rm(path: str):
    try:
        dbutils.fs.rm(path, True)
    except Exception:
        pass


def _safe_ls(path: str, label: str):
    print(f"\n{label}: {path}")
    try:
        for x in dbutils.fs.ls(path):
            print(f" - {x.path}")
    except Exception as e:
        print(f"   [unable to list] {e}")


def publish_data_docs(context, layer_name=PIPELINE_LAYER):
    # Clean local and FileStore targets first so stale HTML does not remain
    _safe_rm(LOCAL_DATA_DOCS_URI)
    _safe_rm(FILESTORE_DATA_DOCS_DBFS_PATH)

    # Build the site on driver local disk, then publish it to FileStore
    context.build_data_docs(site_names=[DATA_DOCS_SITE_NAME])
    dbutils.fs.cp(LOCAL_DATA_DOCS_URI, FILESTORE_DATA_DOCS_DBFS_PATH, True)

    # Verify the expected folders exist
    _safe_ls(FILESTORE_DATA_DOCS_DBFS_PATH, "Data Docs root")
    _safe_ls(FILESTORE_DATA_DOCS_DBFS_PATH + "expectations/", "Data Docs expectations")
    _safe_ls(FILESTORE_DATA_DOCS_DBFS_PATH + "validations/", "Data Docs validations")

    run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    storage_run_path = f"{STORAGE_DATA_DOCS_RUNS_ROOT}{run_stamp}/"

    # Refresh ADLS copies from FileStore
    _safe_rm(STORAGE_DATA_DOCS_LATEST_PATH)
    dbutils.fs.cp(FILESTORE_DATA_DOCS_DBFS_PATH, STORAGE_DATA_DOCS_LATEST_PATH, True)
    dbutils.fs.cp(FILESTORE_DATA_DOCS_DBFS_PATH, storage_run_path, True)

    return {
        "preview_url": FILESTORE_DATA_DOCS_BROWSER_PATH,
        "storage_latest_path": STORAGE_DATA_DOCS_LATEST_PATH,
        "storage_run_path": storage_run_path,
    }


def render_data_docs_preview(data_docs_info, title):
    displayHTML(
        f'''
        <div style="margin:16px 0;">
          <p><strong>{title}</strong></p>
          <p><a href="{data_docs_info["preview_url"]}" target="_blank">Open Data Docs index</a></p>
          <p><a href="/files/great_expectations/bronze/expectations/json_bronze_ingestion_suite.html" target="_blank">Open JSON suite directly</a></p>
          <p><a href="/files/great_expectations/bronze/expectations/csv_bronze_ingestion_suite.html" target="_blank">Open CSV suite directly</a></p>
          <iframe
            src="{data_docs_info["preview_url"]}"
            width="100%"
            height="900"
            style="border:1px solid #d0d7de;border-radius:8px;background:#fff;">
          </iframe>
        </div>
        '''
    )


def ensure_required_columns(df, required_columns, dataset_name):
    missing = [c for c in required_columns if c not in df.columns]
    if missing:
        raise RuntimeError(
            f"Bronze STOP failure for {dataset_name}: missing required columns {missing}"
        )


def empty_issue_df():
    return spark.createDataFrame([], ISSUE_SCHEMA)


def issue_events_from_condition(df, condition, dataset, rule_id, severity, dq_reason):
    out_df = df.filter(condition)
    if "_source_file" not in out_df.columns:
        out_df = out_df.withColumn("_source_file", F.lit(None).cast("string"))
    if "InvoiceId" not in out_df.columns:
        out_df = out_df.withColumn("InvoiceId", F.lit(None).cast("string"))
    return (
        out_df.select(
            F.lit("bronze").alias("layer"),
            F.lit(dataset).alias("dataset"),
            F.lit(rule_id).alias("rule_id"),
            F.lit(severity).alias("severity"),
            F.col("_source_file").cast("string").alias("_source_file"),
            F.col("InvoiceId").cast("string").alias("InvoiceId"),
            F.lit(None).cast("string").alias("LineNumber"),
            F.lit(dq_reason).alias("dq_reason"),
            F.current_timestamp().alias("issue_ts"),
        ).dropDuplicates()
    )


def extract_first_validation(checkpoint_result):
    run_results = checkpoint_result.run_results
    first_key = list(run_results.keys())[0]
    return run_results[first_key]["validation_result"]


def summarize_result(validation_result, suite_name):
    stats = validation_result.get("statistics", {}) or {}
    return {
        "suite": suite_name,
        "success": validation_result.get("success"),
        "evaluated": stats.get("evaluated_expectations", 0),
        "successful": stats.get("successful_expectations", 0),
        "failed": stats.get("unsuccessful_expectations", 0),
    }


def display_summary(summary):
    print("=" * 65)
    print(f"Suite   : {summary['suite']}")
    print(f"Status  : {'PASSED' if summary['success'] else 'FAILED'}")
    print(
        f"Evaluated: {summary['evaluated']} | "
        f"Passed: {summary['successful']} | Failed: {summary['failed']}"
    )
    print("=" * 65)

In [0]:
# Runtime metrics recording helper functions
def first_validation_result(checkpoint_result):
    run_results = checkpoint_result.run_results
    return run_results[list(run_results.keys())[0]]["validation_result"]


def validation_statistics(validation_result):
    stats = validation_result.get("statistics", {}) or {}
    evaluated = int(stats.get("evaluated_expectations", 0) or 0)
    successful = int(stats.get("successful_expectations", 0) or 0)
    failed = int(stats.get("unsuccessful_expectations", 0) or 0)
    success_percent = float(successful / evaluated) if evaluated else None
    return evaluated, successful, failed, success_percent


def write_ge_runtime_metric(layer, suite_name, run_id, started_at, ended_at, rows_evaluated, validation_result):
    runtime_seconds = (ended_at - started_at).total_seconds()
    evaluated, successful, failed, success_percent = validation_statistics(validation_result)
    rows_per_second = float(rows_evaluated / runtime_seconds) if rows_evaluated and runtime_seconds > 0 else None

    row = [(
        layer,
        suite_name,
        run_id,
        started_at,
        ended_at,
        float(runtime_seconds),
        int(rows_evaluated) if rows_evaluated is not None else None,
        rows_per_second,
        evaluated,
        successful,
        failed,
        success_percent,
        bool(validation_result.get("success")),
        datetime.now(timezone.utc),
    )]

    runtime_df = spark.createDataFrame(row, GE_RUNTIME_SCHEMA)
    runtime_df.write.format("delta").mode("append").save(GE_RUNTIME_METRICS_PATH)

    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG_NAME}`.{SCHEMA_NAME}")
    spark.sql(f'''
        CREATE TABLE IF NOT EXISTS {GE_RUNTIME_METRICS_TABLE}
        USING DELTA
        LOCATION "{GE_RUNTIME_METRICS_PATH}"
    ''')

In [0]:
csv_bronze_df = spark.read.format("delta").load(CSV_BRONZE_PATH)
json_bronze_df = spark.read.format("delta").load(JSON_BRONZE_PATH)

ensure_required_columns(csv_bronze_df, REQUIRED_CSV_COLUMNS, "csv_raw")
ensure_required_columns(json_bronze_df, REQUIRED_JSON_COLUMNS, "json_raw")

if csv_bronze_df.rdd.isEmpty():
    raise RuntimeError("Bronze STOP failure: csv_bronze is empty after ingest.")
if json_bronze_df.rdd.isEmpty():
    raise RuntimeError("Bronze STOP failure: json_bronze is empty after ingest.")

print(f"csv_bronze_df rows  = {csv_bronze_df.count():,}")
print(f"json_bronze_df rows = {json_bronze_df.count():,}")

csv_bronze_df rows  = 1,000
json_bronze_df rows = 1,007


In [0]:
bronze_ge_context = create_gx_context()
bronze_datasource = add_or_update_spark_datasource(bronze_ge_context, "bronze_runtime_datasource")


def run_csv_bronze_validation(df):
    suite_name = "csv_bronze_ingestion_suite"
    batch_request = add_or_update_dataframe_asset(bronze_datasource, "csv_bronze_ingestion_asset", df)
    bronze_ge_context.add_or_update_expectation_suite(expectation_suite_name=suite_name)
    validator = bronze_ge_context.get_validator(batch_request=batch_request, expectation_suite_name=suite_name)
    validator.expect_table_row_count_to_be_between(min_value=1, max_value=None)
    validator.expect_column_values_to_not_be_null("_source_file")
    validator.expect_column_values_to_not_be_null("_ingest_ts")
    if "_rescued_data" in df.columns:
        validator.expect_column_values_to_be_null("_rescued_data")
    validator.save_expectation_suite(discard_failed_expectations=False)
    checkpoint_name = "csv_bronze_ingestion_checkpoint"
    build_dataframe_checkpoint(bronze_ge_context, batch_request, suite_name, checkpoint_name)
    start_at = datetime.now(timezone.utc)
    checkpoint_result = bronze_ge_context.run_checkpoint(checkpoint_name=checkpoint_name)
    ended_at = datetime.now(timezone.utc)
    validation_result = extract_first_validation(checkpoint_result)
    write_ge_runtime_metric(
        layer="bronze",
        suite_name=suite_name,
        run_id=checkpoint_result.run_id,
        started_at=start_at,
        ended_at=ended_at,
        rows_evaluated=df.count(),
        validation_result=validation_result
    )
    return checkpoint_result, suite_name


def run_json_bronze_validation(df):
    suite_name = "json_bronze_ingestion_suite"
    batch_request = add_or_update_dataframe_asset(bronze_datasource, "json_bronze_ingestion_asset", df)
    bronze_ge_context.add_or_update_expectation_suite(expectation_suite_name=suite_name)
    validator = bronze_ge_context.get_validator(batch_request=batch_request, expectation_suite_name=suite_name)
    validator.expect_table_row_count_to_be_between(min_value=1, max_value=None)
    validator.expect_column_values_to_not_be_null("_source_file")
    validator.expect_column_values_to_be_unique("_source_file")
    validator.expect_column_values_to_not_be_null("_ingest_ts")
    validator.expect_column_values_to_not_be_null("analyzeResult")
    if "_rescued_data" in df.columns:
        validator.expect_column_values_to_be_null("_rescued_data")
    validator.save_expectation_suite(discard_failed_expectations=False)
    checkpoint_name = "json_bronze_ingestion_checkpoint"
    build_dataframe_checkpoint(bronze_ge_context, batch_request, suite_name, checkpoint_name)
    start_at = datetime.now(timezone.utc)
    checkpoint_result = bronze_ge_context.run_checkpoint(checkpoint_name=checkpoint_name)
    ended_at = datetime.now(timezone.utc)
    validation_result = extract_first_validation(checkpoint_result)
    write_ge_runtime_metric(
        layer="bronze",
        suite_name=suite_name,
        run_id=checkpoint_result.run_id,
        started_at=start_at,
        ended_at=ended_at,
        rows_evaluated=df.count(),
        validation_result=validation_result
    )
    return checkpoint_result, suite_name


csv_result, csv_suite_name = run_csv_bronze_validation(csv_bronze_df)
json_result, json_suite_name = run_json_bronze_validation(json_bronze_df)

for summary in [
    summarize_result(extract_first_validation(csv_result), csv_suite_name),
    summarize_result(extract_first_validation(json_result), json_suite_name),
]:
    display_summary(summary)

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/22 [00:00<?, ?it/s]

...

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/33 [00:00<?, ?it/s]

Suite   : csv_bronze_ingestion_suite
Status  : PASSED
Evaluated: 4 | Passed: 4 | Failed: 0
Suite   : json_bronze_ingestion_suite
Status  : PASSED
Evaluated: 6 | Passed: 6 | Failed: 0


In [0]:
csv_issue_events_df = empty_issue_df()
csv_issue_events_df = csv_issue_events_df.unionByName(issue_events_from_condition(
    csv_bronze_df,
    F.col("_source_file").isNull() | (F.trim(F.col("_source_file")) == ""),
    "csv_raw",
    "bronze_csv_source_file_present",
    "STOP",
    "Source file lineage is missing in Bronze CSV",
), allowMissingColumns=True)
csv_issue_events_df = csv_issue_events_df.unionByName(issue_events_from_condition(
    csv_bronze_df,
    F.col("_ingest_ts").isNull(),
    "csv_raw",
    "bronze_csv_ingest_ts_present",
    "STOP",
    "Ingest timestamp is missing in Bronze CSV",
), allowMissingColumns=True)
if "_rescued_data" in csv_bronze_df.columns:
    csv_issue_events_df = csv_issue_events_df.unionByName(issue_events_from_condition(
        csv_bronze_df,
        F.col("_rescued_data").isNotNull() & (F.trim(F.col("_rescued_data")) != ""),
        "csv_raw",
        "bronze_csv_rescued_data_empty",
        "ERROR",
        "Auto Loader rescued malformed CSV content",
    ), allowMissingColumns=True)

json_issue_events_df = empty_issue_df()
json_issue_events_df = json_issue_events_df.unionByName(issue_events_from_condition(
    json_bronze_df,
    F.col("_source_file").isNull() | (F.trim(F.col("_source_file")) == ""),
    "json_raw",
    "bronze_json_source_file_present",
    "STOP",
    "Source file lineage is missing in Bronze JSON",
), allowMissingColumns=True)
json_issue_events_df = json_issue_events_df.unionByName(issue_events_from_condition(
    json_bronze_df,
    F.col("_ingest_ts").isNull(),
    "json_raw",
    "bronze_json_ingest_ts_present",
    "STOP",
    "Ingest timestamp is missing in Bronze JSON",
), allowMissingColumns=True)
json_issue_events_df = json_issue_events_df.unionByName(issue_events_from_condition(
    json_bronze_df,
    F.col("analyzeResult").isNull() | (F.trim(F.col("analyzeResult").cast("string")) == ""),
    "json_raw",
    "bronze_json_analyze_result_present",
    "ERROR",
    "Document Intelligence payload is missing",
), allowMissingColumns=True)
if "status" in json_bronze_df.columns:
    json_issue_events_df = json_issue_events_df.unionByName(issue_events_from_condition(
        json_bronze_df,
        F.col("status").isNotNull() & (F.lower(F.col("status")) != "succeeded"),
        "json_raw",
        "bronze_json_status_succeeded",
        "WARNING",
        "Document Intelligence status is not succeeded",
    ), allowMissingColumns=True)
if "_rescued_data" in json_bronze_df.columns:
    json_issue_events_df = json_issue_events_df.unionByName(issue_events_from_condition(
        json_bronze_df,
        F.col("_rescued_data").isNotNull() & (F.trim(F.col("_rescued_data")) != ""),
        "json_raw",
        "bronze_json_rescued_data_empty",
        "ERROR",
        "Auto Loader rescued malformed JSON content",
    ), allowMissingColumns=True)

json_duplicate_source_files_df = json_bronze_df.groupBy("_source_file").count().filter(F.col("count") > 1).select("_source_file")
json_duplicate_issue_events_df = (
    json_bronze_df.join(json_duplicate_source_files_df, on="_source_file", how="inner")
    .select(
        F.lit("bronze").alias("layer"),
        F.lit("json_raw").alias("dataset"),
        F.lit("bronze_json_duplicate_source_file").alias("rule_id"),
        F.lit("ERROR").alias("severity"),
        F.col("_source_file").cast("string").alias("_source_file"),
        F.lit(None).cast("string").alias("InvoiceId"),
        F.lit(None).cast("string").alias("LineNumber"),
        F.lit("JSON source file was ingested more than once").alias("dq_reason"),
        F.current_timestamp().alias("issue_ts"),
    ).dropDuplicates()
)

bronze_issue_events_df = (
    csv_issue_events_df
    .unionByName(json_issue_events_df, allowMissingColumns=True)
    .unionByName(json_duplicate_issue_events_df, allowMissingColumns=True)
    .dropDuplicates()
)
display(bronze_issue_events_df.orderBy("dataset", "severity", "rule_id"))

layer,dataset,rule_id,severity,_source_file,InvoiceId,LineNumber,dq_reason,issue_ts


In [0]:
bronze_stop_failures = bronze_issue_events_df.filter(F.col("severity") == "STOP").count()
if bronze_stop_failures > 0:
    display(bronze_issue_events_df.filter(F.col("severity") == "STOP"))
    raise RuntimeError(f"Bronze STOP failures detected ({bronze_stop_failures}). Downstream processing halted.")

csv_error_keys_df = (
    bronze_issue_events_df
    .filter((F.col("dataset") == "csv_raw") & (F.col("severity") == "ERROR"))
    .select("_source_file")
    .distinct()
)
csv_warning_keys_df = (
    bronze_issue_events_df
    .filter((F.col("dataset") == "csv_raw") & (F.col("severity") == "WARNING"))
    .select("_source_file")
    .distinct()
)
json_error_keys_df = (
    bronze_issue_events_df
    .filter((F.col("dataset") == "json_raw") & (F.col("severity") == "ERROR"))
    .select("_source_file")
    .distinct()
)
json_warning_keys_df = (
    bronze_issue_events_df
    .filter((F.col("dataset") == "json_raw") & (F.col("severity") == "WARNING"))
    .select("_source_file")
    .distinct()
)

valid_csv_bronze_df = csv_bronze_df.join(csv_error_keys_df, on="_source_file", how="left_anti")
quarantine_csv_df = csv_bronze_df.join(csv_error_keys_df, on="_source_file", how="inner")
warning_csv_df = csv_bronze_df.join(csv_warning_keys_df, on="_source_file", how="inner")

valid_json_bronze_df = json_bronze_df.join(json_error_keys_df, on="_source_file", how="left_anti")
quarantine_json_df = json_bronze_df.join(json_error_keys_df, on="_source_file", how="inner")
warning_json_df = json_bronze_df.join(json_warning_keys_df, on="_source_file", how="inner")

valid_csv_bronze_df.write.format("delta").mode("overwrite").save(VALID_CSV_BRONZE_PATH)
valid_json_bronze_df.write.format("delta").mode("overwrite").save(VALID_JSON_BRONZE_PATH)
quarantine_csv_df.write.format("delta").mode("overwrite").save(QUARANTINE_CSV_BRONZE_PATH)
quarantine_json_df.write.format("delta").mode("overwrite").save(QUARANTINE_JSON_BRONZE_PATH)
warning_csv_df.write.format("delta").mode("overwrite").save(WARNING_CSV_BRONZE_PATH)
warning_json_df.write.format("delta").mode("overwrite").save(WARNING_JSON_BRONZE_PATH)
bronze_issue_events_df.write.format("delta").mode("overwrite").save(BRONZE_ISSUE_LOG_PATH)

# Configure Data Docs site BEFORE publishing; publish_data_docs copies it to FileStore
ensure_data_docs_site(bronze_ge_context)

bronze_data_docs = publish_data_docs(bronze_ge_context)
render_data_docs_preview(bronze_data_docs, "Bronze Great Expectations Data Docs")

dbutils.jobs.taskValues.set(key="valid_csv_path", value=VALID_CSV_BRONZE_PATH)
dbutils.jobs.taskValues.set(key="valid_json_path", value=VALID_JSON_BRONZE_PATH)
dbutils.jobs.taskValues.set(key="bronze_issue_log_path", value=BRONZE_ISSUE_LOG_PATH)


Data Docs root: dbfs:/FileStore/great_expectations/bronze/
 - dbfs:/FileStore/great_expectations/bronze/expectations/
 - dbfs:/FileStore/great_expectations/bronze/index.html
 - dbfs:/FileStore/great_expectations/bronze/static/
 - dbfs:/FileStore/great_expectations/bronze/validations/

Data Docs expectations: dbfs:/FileStore/great_expectations/bronze/expectations/
 - dbfs:/FileStore/great_expectations/bronze/expectations/csv_bronze_ingestion_suite.html
 - dbfs:/FileStore/great_expectations/bronze/expectations/json_bronze_ingestion_suite.html

Data Docs validations: dbfs:/FileStore/great_expectations/bronze/validations/
 - dbfs:/FileStore/great_expectations/bronze/validations/csv_bronze_ingestion_suite/
 - dbfs:/FileStore/great_expectations/bronze/validations/json_bronze_ingestion_suite/


Bronze Great Expectations Data Docs 
 Open Data Docs index 
 Open JSON suite directly 
 Open CSV suite directly

In [0]:
# Databricks table registration for Metaplane: Bronze GE outputs
CATALOG_NAME = "hant-catalog"
SCHEMA_NAME = "invoice"

BATCH_BRONZE_VALID_CSV_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.batch_bronze_valid_csv_raw"
BATCH_BRONZE_VALID_JSON_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.batch_bronze_valid_json_raw"
BATCH_BRONZE_QUARANTINE_CSV_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.batch_bronze_quarantine_csv_raw"
BATCH_BRONZE_QUARANTINE_JSON_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.batch_bronze_quarantine_json_raw"
BATCH_BRONZE_WARNING_CSV_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.batch_bronze_warning_csv_raw"
BATCH_BRONZE_WARNING_JSON_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.batch_bronze_warning_json_raw"
BATCH_BRONZE_GE_ISSUE_LOG_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.batch_bronze_ge_issue_log"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG_NAME}`.{SCHEMA_NAME}")

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {BATCH_BRONZE_VALID_CSV_TABLE}
USING DELTA
LOCATION "{VALID_CSV_BRONZE_PATH}"
''')

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {BATCH_BRONZE_VALID_JSON_TABLE}
USING DELTA
LOCATION "{VALID_JSON_BRONZE_PATH}"
''')

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {BATCH_BRONZE_QUARANTINE_CSV_TABLE}
USING DELTA
LOCATION "{QUARANTINE_CSV_BRONZE_PATH}"
''')

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {BATCH_BRONZE_QUARANTINE_JSON_TABLE}
USING DELTA
LOCATION "{QUARANTINE_JSON_BRONZE_PATH}"
''')

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {BATCH_BRONZE_WARNING_CSV_TABLE}
USING DELTA
LOCATION "{WARNING_CSV_BRONZE_PATH}"
''')

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {BATCH_BRONZE_WARNING_JSON_TABLE}
USING DELTA
LOCATION "{WARNING_JSON_BRONZE_PATH}"
''')

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {BATCH_BRONZE_GE_ISSUE_LOG_TABLE}
USING DELTA
LOCATION "{BRONZE_ISSUE_LOG_PATH}"
''')

display(spark.sql(f"DESCRIBE DETAIL {BATCH_BRONZE_GE_ISSUE_LOG_TABLE}"))


format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,16139b1c-92a9-4ce4-8d5c-efda3ca50e21,hant-catalog.invoice.batch_bronze_ge_issue_log,null,abfss://lakehouse@hantstorageaccount.dfs.core.windows.net/bronze/ge/issue_log,2026-04-23T18:41:04.745Z,2026-04-23T18:41:12Z,List(),List(),0,0,Map(delta.enableDeletionVectors -> true),3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false
